# Example 7.11.2 / 1: Capacitated Facility Location

Companion notebook for **7.11.2 Facility Location** in the AMPL lab Documentation, section "Capacitated Facility Location". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Capacitated Facility Location

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set FACILITIES;                         # candidate facility sites
set CUSTOMERS;                          # customers to serve
param fixed_cost {FACILITIES} >= 0;     # cost of opening the facility
param ship_cost {FACILITIES, CUSTOMERS} >= 0;  # cost per unit shipped
param capacity {FACILITIES} >= 0;       # units an open facility can ship
param demand {CUSTOMERS} >= 0;          # units each customer requires

var Open {FACILITIES} binary;           # 1 if the facility is opened
var Ship {FACILITIES, CUSTOMERS} >= 0;  # units shipped to the customer

# fixed opening costs plus shipping costs
minimize TotalCost:
    sum {i in FACILITIES} fixed_cost[i] * Open[i]
  + sum {i in FACILITIES, j in CUSTOMERS} ship_cost[i,j] * Ship[i,j];
# each customer's demand is met exactly
subject to Serve {j in CUSTOMERS}:
    sum {i in FACILITIES} Ship[i,j] = demand[j];
# only an open facility ships, up to its capacity
subject to Capacity {i in FACILITIES}:
    sum {j in CUSTOMERS} Ship[i,j] <= capacity[i] * Open[i];

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["FACILITIES"] = ["north", "center", "south"]
ampl.set["CUSTOMERS"] = ["c1", "c2", "c3", "c4"]
ampl.param["fixed_cost"] = {"north": 400, "center": 500, "south": 300}
ampl.param["capacity"] = {"north": 60, "center": 70, "south": 50}
ampl.param["demand"] = {"c1": 20, "c2": 30, "c3": 25, "c4": 15}
ampl.param["ship_cost"] = {
    ("north", "c1"): 2, ("north", "c2"): 4, ("north", "c3"): 7, ("north", "c4"): 9,
    ("center", "c1"): 5, ("center", "c2"): 3, ("center", "c3"): 3, ("center", "c4"): 5,
    ("south", "c1"): 9, ("south", "c2"): 7, ("south", "c3"): 4, ("south", "c4"): 2}

In [5]:
# run.py
ampl.solve(solver="highs")
print("open:", [i for i, v in ampl.var["Open"].to_dict().items() if v > 0.5])
print(ampl.var["Ship"].to_pandas().query("`Ship.val` > 0"))

HiGHS 1.14.0: optimal solution; objective 990
35 simplex iterations
4 branching nodes
open: ['north', 'south']
               Ship.val
index0 index1          
north  c1            20
       c2            30
south  c3            25
       c4            15


### Pure AMPL: data.dat and model.run

The customers need 90 units and no single site can supply them all, so two sites open. North and
south, the cheapest pair to open, each serve their nearby customers: 700 in fixed costs plus 290
in shipping.

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
param: FACILITIES: fixed_cost capacity :=
  north   400  60
  center  500  70
  south   300  50 ;
param: CUSTOMERS: demand := c1 20  c2 30  c3 25  c4 15;
param ship_cost:  c1 c2 c3 c4 :=
  north            2  4  7  9
  center           5  3  3  5
  south            9  7  4  2 ;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option solver highs;
solve;
display Open;
display {i in FACILITIES, j in CUSTOMERS: Ship[i,j] > 0} Ship[i,j];

HiGHS 1.14.0: optimal solution; objective 990
35 simplex iterations
4 branching nodes
Open [*] :=
center  0
 north  1
 south  1
;

Ship[i,j] :=
north c1   20
north c2   30
south c3   25
south c4   15
;

